In [16]:
!nvidia-smi 

Thu Mar 26 01:26:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             31W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
!python --version

Python 3.12.12


In [18]:
!git clone https://github.com/KhoiTrant68/WMDC.git

Cloning into 'WMDC'...
remote: Enumerating objects: 919, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 919 (delta 150), reused 15 (delta 9), pack-reused 688 (from 1)
Receiving objects: 100% (919/919), 453.05 KiB | 5.27 MiB/s, done.
Resolving deltas: 100% (578/578), done.


In [19]:
!pip install compressai accelerate einops timm tensorboard pytorch-msssim flops-profiler

In [20]:
!pip install tensorboard==2.14
!pip install protobuf==4.25.3


In [21]:
%cd /kaggle/working/WMDC/vmamba
!pip install .

/kaggle/working/WMDC/vmamba
Processing /kaggle/working/WMDC/vmamba
  Preparing metadata (setup.py) ... done
  Created wheel for selective_scan: filename=selective_scan-0.0.2-cp312-cp312-linux_x86_64.whl size=16758465 sha256=70cd5fad5135507a3b889dacbc36ffea50f4980ff87b77a74dd2e4a1fab65e17
  Stored in directory: /tmp/pip-ephem-wheel-cache-kylz_pgi/wheels/cf/c6/d2/8cefb039567092132dd5bd5787cada54127ff9ba21fc29d5ec
Successfully built selective_scan
  Attempting uninstall: selective_scan
    Found existing installation: selective_scan 0.0.2
    Uninstalling selective_scan-0.0.2:
      Successfully uninstalled selective_scan-0.0.2


In [22]:
%cd /kaggle/working/WMDC

/kaggle/working/WMDC


In [40]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.50 KiB | 1.50 MiB/s, done.
From https://github.com/KhoiTrant68/WMDC
   bb51ee1..ff82a48  main       -> origin/main
Updating bb51ee1..ff82a48
Fast-forward
 analyze/visualize_attention.py | 222 +++++++++++++++++++----------------------
 1 file changed, 103 insertions(+), 119 deletions(-)


In [24]:
!ls

analyze			    format_code.bash  run-wmdc.ipynb
assets			    LICENSE	      train.py
checkpoints_balanced_eot    models	      vmamba
checkpoints_softmax	    modules	      WMDC
checkpoints_unbalanced_eot  README.md
eval.py			    requirements.txt


In [25]:
!rm -rf visual*
!rm -rf hdda*
!rm -rf checkpoints*
!rm -rf kodak*
!rm -rf latent*

In [26]:
%cd /kaggle/working/WMDC
# !accelerate launch train.py -d /kaggle/input/datasets/tranjohan/data-1000-test/dataset_1000_test --save_path checkpoints_softmax --epochs 2 --batch-size 16 --lambda 0.0018 --routing_mode softmax
# !accelerate launch train.py -d /kaggle/input/datasets/tranjohan/data-1000-test/dataset_1000_test --save_path checkpoints_balanced_eot --epochs 2 --batch-size 16 --lambda 0.0018 --routing_mode balanced_eot
# !accelerate launch train.py -d /kaggle/input/datasets/tranjohan/data-1000-test/dataset_1000_test --save_path checkpoints_unbalanced_eot --epochs 2 --batch-size 16 --lambda 0.0018 --routing_mode unbalanced_eot
!accelerate launch train.py -d /kaggle/input/datasets/khitrnminh/dataset-20000-test/dataset --save_path checkpoints_softmax --epochs 4 --batch-size 8 --lambda 0.0018 --routing_mode softmax
!accelerate launch train.py -d /kaggle/input/datasets/khitrnminh/dataset-20000-test/dataset --save_path checkpoints_balanced_eot --epochs 4 --batch-size 8 --lambda 0.0018 --routing_mode balanced_eot
!accelerate launch train.py -d /kaggle/input/datasets/khitrnminh/dataset-20000-test/dataset --save_path checkpoints_unbalanced_eot --epochs 4 --batch-size 8 --lambda 0.0018 --routing_mode unbalanced_eot



/kaggle/working/WMDC
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-03-26 01:27:39.768279: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774488459.793388    1264 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774488459.802390    1264 cuda_blas.cc:1407] Unable to register cuBLAS factory

In [27]:
%cd /kaggle/working/WMDC
!python eval.py --dataset /kaggle/input/datasets/khitrnminh/kodak-test --checkpoint /kaggle/working/WMDC/checkpoints_softmax/lambda_0.0018_mse/checkpoint_best.pth.tar --output kodak_softmax --cuda --routing_mode softmax --measure-dict-util 
!python eval.py --dataset /kaggle/input/datasets/khitrnminh/kodak-test --checkpoint /kaggle/working/WMDC/checkpoints_balanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar --output kodak_balanced_eot --cuda --routing_mode balanced_eot --measure-dict-util 
!python eval.py --dataset /kaggle/input/datasets/khitrnminh/kodak-test --checkpoint /kaggle/working/WMDC/checkpoints_unbalanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar --output kodak_unbalanced_eot --cuda --routing_mode unbalanced_eot --measure-dict-util


/kaggle/working/WMDC
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
kodim01.png                    | BPP: 0.3181 (pad: 0.3181, oh: +0.00000) | PSNR: 17.44 dB | MS-SSIM: 0.4032 | Enc: 0.763s  Dec: 0.279s | Util: [100.0%, 100.0%, 100.0%, 100.0%, 100.0%] mean=100.0%
kodim02.png                    | BPP: 0.2854 (pad: 0.2854, oh: +0.00000) | PSNR: 14.48 dB | MS-SSIM: 0.6101 | Enc: 0.267s  Dec: 0.208s | Util: [100.0%, 100.0%, 100.0%, 100.0%, 100.0%] mean=100.0%
kodim03.png                    | BPP: 0.3172 (pad: 0.3172, oh: +0.00000) | PSNR: 17.27 dB | MS-SSIM: 0.6422 | Enc: 0.273s  Dec: 0.210s | Util: [100.0%, 100.0%, 100.0%, 100.0%, 100.0%] mean=100.0%
kodim04.png                    | BPP: 0.3354 (pad: 0.3354, oh: +0.00000) | PSNR: 17.32 dB | MS-SSIM: 0.6075 | Enc: 0.288s  Dec: 0.209s | Util: [100.0%, 100.0%, 100.0%, 100.0%, 100

In [41]:
%cd /kaggle/working/WMDC

!python analyze/visualize_attention.py \
    --img_dir /kaggle/input/datasets/khitrnminh/kodak-test \
    --checkpoint /kaggle/working/WMDC/checkpoints_softmax/lambda_0.0018_mse/checkpoint_best.pth.tar \
    --routing_mode softmax \
    --mode top_tokens \
    --slice 4 \
    --cuda \
    --output hdda_attention_maps_softmax.pdf

!python analyze/visualize_attention.py \
    --img_dir /kaggle/input/datasets/khitrnminh/kodak-test \
    --checkpoint /kaggle/working/WMDC/checkpoints_balanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar \
    --routing_mode balanced_eot \
    --mode top_tokens \
    --slice 4 \
    --cuda \
    --output hdda_attention_maps_balanced_eot.pdf

!python analyze/visualize_attention.py \
    --img_dir /kaggle/input/datasets/khitrnminh/kodak-test \
    --checkpoint /kaggle/working/WMDC/checkpoints_unbalanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar \
    --routing_mode unbalanced_eot \
    --mode top_tokens \
    --slice 4 \
    --cuda \
    --output hdda_attention_maps_unbalanced_eot.pdf

!python analyze/visualize_attention.py \
    --img_dir /kaggle/input/datasets/khitrnminh/kodak-test \
    --checkpoint /kaggle/working/WMDC/checkpoints_softmax/lambda_0.0018_mse/checkpoint_best.pth.tar \
    --routing_mode softmax \
    --mode slice_evolution \
    --target_token 42 \
    --cuda \
    --output hdda_slice_evolution_softmax.pdf

!python analyze/visualize_attention.py \
    --img_dir /kaggle/input/datasets/khitrnminh/kodak-test \
    --checkpoint /kaggle/working/WMDC/checkpoints_balanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar \
    --routing_mode balanced_eot \
    --mode slice_evolution \
    --target_token 42 \
    --cuda \
    --output hdda_slice_evolution_balanced_eot.pdf


!python analyze/visualize_attention.py \
    --img_dir /kaggle/input/datasets/khitrnminh/kodak-test \
    --checkpoint /kaggle/working/WMDC/checkpoints_unbalanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar \
    --routing_mode unbalanced_eot \
    --mode slice_evolution \
    --target_token 42 \
    --cuda \
    --output hdda_slice_evolution_unbalanced_eot.pdf

/kaggle/working/WMDC
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Processing kodim01.png...
  Top 4 tokens for kodim01.png (Slice 4): [104, 93, 92, 48]
Processing kodim02.png...
  Top 4 tokens for kodim02.png (Slice 4): [104, 28, 93, 66]
Processing kodim03.png...
  Top 4 tokens for kodim03.png (Slice 4): [104, 93, 92, 28]
Processing kodim04.png...
  Top 4 tokens for kodim04.png (Slice 4): [104, 93, 28, 92]

Saved attention maps to hdda_attention_maps_softmax.pdf
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Processing kodim01.png...
  Top 4 tokens for kodim01.png (Slice 4): [37, 7, 71, 11]
Processing kodim02.png...
  Top 4 tokens for kodim02.png (Slice 4): [1

In [42]:
%cd /kaggle/working/WMDC
!python analyze/visualize_patches.py -i /kaggle/input/datasets/khitrnminh/kodak-test/kodim01.png -c /kaggle/working/WMDC/checkpoints_softmax/lambda_0.0018_mse/checkpoint_best.pth.tar --cuda --routing_mode softmax -o visual_comparison_softmax
!python analyze/visualize_patches.py -i /kaggle/input/datasets/khitrnminh/kodak-test/kodim01.png -c /kaggle/working/WMDC/checkpoints_balanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar --cuda --routing_mode balanced_eot -o visual_comparison_balanced_eot
!python analyze/visualize_patches.py -i /kaggle/input/datasets/khitrnminh/kodak-test/kodim01.png -c /kaggle/working/WMDC/checkpoints_unbalanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar --cuda --routing_mode unbalanced_eot -o visual_comparison_unbalanced_eot


/kaggle/working/WMDC
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Saved visual_comparison_softmax.pdf
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Saved visual_comparison_balanced_eot.pdf
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Saved visual_comparison_unbalanced_eot.pdf


In [43]:
%cd /kaggle/working/WMDC
!python analyze/visualize_latents.py --image /kaggle/input/datasets/khitrnminh/kodak-test/kodim01.png --checkpoint /kaggle/working/WMDC/checkpoints_softmax/lambda_0.0018_mse/checkpoint_best.pth.tar --routing_mode softmax --output latent_sparsity_visualization_softmax.pdf
!python analyze/visualize_latents.py --image /kaggle/input/datasets/khitrnminh/kodak-test/kodim01.png --checkpoint /kaggle/working/WMDC/checkpoints_balanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar --routing_mode balanced_eot --output latent_sparsity_visualization_balanced_eot.pdf
!python analyze/visualize_latents.py --image /kaggle/input/datasets/khitrnminh/kodak-test/kodim01.png --checkpoint /kaggle/working/WMDC/checkpoints_unbalanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar --routing_mode unbalanced_eot --output latent_sparsity_visualization_unbalanced_eot.pdf


/kaggle/working/WMDC
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Saved latent visualisation to latent_sparsity_visualization_softmax.pdf
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Saved latent visualisation to latent_sparsity_visualization_balanced_eot.pdf
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Saved latent visualisation to latent_sparsity_visualization_unbalanced_eot.pdf


In [44]:
# %cd /kaggle/working/WMDC
# !python analyze/ablation_sinkhorn_convergence.py \
#         --checkpoint /kaggle/working/WMDC/checkpoints_unbalanced_eot/lambda_0.0018_mse/checkpoint_best.pth.tar \
#         --image /kaggle/input/datasets/khitrnminh/kodak-test/kodim04.png \
#         --routing_mode unbalanced_eot \
#         -o sinkhorn_convergence_unbalanced_eot.pdf \
#         --cuda

/kaggle/working/WMDC
/kaggle/working/WMDC/modules/VSS_module.py:29: UserWarning: Failed to import selective_scan_cuda. Error: No module named 'selective_scan_cuda'
  warnings.warn(f"Failed to import selective_scan_cuda. Error: {e}")
Traceback (most recent call last):
  File "/kaggle/working/WMDC/analyze/ablation_sinkhorn_convergence.py", line 185, in <module>
    main()
  File "/kaggle/working/WMDC/analyze/ablation_sinkhorn_convergence.py", line 143, in main
    iter_counts, costs = measure_convergence(model, x_padded, args.max_iters, device)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/WMDC/analyze/ablation_sinkhorn_convergence.py", line 57, in measure_convergence
    C_mat = model.eot_attention._cost_matrix(query, k_dict, H, W)
            ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1965, in __getattr__
    raise AttributeError(
AttributeError: 'WMDC' object has 

In [32]:
!ls /kaggle/working/WMDC/checkpoints_unbalanced_eot/lambda_0.0018_mse

checkpoint_best.pth.tar    tb
checkpoint_latest.pth.tar  train_20260326_013404.log


In [33]:
!rm -rf result_save*

In [34]:
!mkdir result_save


In [47]:
# !mv visual*  result_save
!mv *.pdf result_save
# !mv kodak* result_save
# !mv latent* result_save

mv: cannot stat '*.pdf': No such file or directory


In [36]:
!du -sh *

72K	analyze
20K	assets
1.2G	checkpoints_balanced_eot
1.2G	checkpoints_softmax
1.2G	checkpoints_unbalanced_eot
16K	eval.py
4.0K	format_code.bash
12K	LICENSE
52K	models
216K	modules
8.0K	README.md
4.0K	requirements.txt
57M	result_save
96K	run-wmdc.ipynb
24K	train.py
104M	vmamba
1.4M	WMDC


In [48]:
!zip -r result_save.zip result_save

updating: result_save/ (stored 0%)
updating: result_save/latent_sparsity_visualization_unbalanced_eot.pdf (deflated 6%)
updating: result_save/visual_comparison_unbalanced_eot.pdf (deflated 13%)
updating: result_save/kodak_unbalanced_eot/ (stored 0%)
updating: result_save/kodak_unbalanced_eot/RD_report.json (deflated 91%)
updating: result_save/kodak_unbalanced_eot/images/ (stored 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim09.png (deflated 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim20.png (deflated 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim14.png (deflated 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim01.png (deflated 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim24.png (deflated 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim06.png (deflated 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim21.png (deflated 0%)
updating: result_save/kodak_unbalanced_eot/images/kodim15.png (deflated 0%)
updati